# Лаб 5 вариант 3

## Задание 1 
В примере алгоритма поиска в ширину из лекции
(https://cloud.mail.ru/public/9yCy/LitbGAfEg) структура данных граф представлена
словарем. Переделайте этот пример, заменив словарь на реализацию класса Graph из
лекционных материалов.

In [2]:
from collections import deque

class Vertex:
    def __init__(self, key):
        self.id = key
        self.connectedTo = {}
        self.color = 'white'   # white - не посещён, grey - в очереди, black - посещён
        self.dist = 0
        self.pred = None
    
    def addNeighbor(self, nbr, weight=0):
        self.connectedTo[nbr] = weight
    
    def getConnections(self):
        return self.connectedTo.keys()
    
    def getId(self):
        return self.id


class Graph:
    def __init__(self):
        self.vertList = {}
        self.numVertices = 0
    
    def addVertex(self, key):
        self.numVertices += 1
        newVertex = Vertex(key)
        self.vertList[key] = newVertex
        return newVertex
    
    def getVertex(self, n):
        return self.vertList.get(n)
    
    def addEdge(self, f, t, cost=0):
        if f not in self.vertList:
            self.addVertex(f)
        if t not in self.vertList:
            self.addVertex(t)
        self.vertList[f].addNeighbor(self.vertList[t], cost)
    
    def getVertices(self):
        return self.vertList.keys()
    
    def __iter__(self):
        return iter(self.vertList.values())


def bfs(graph, start):
    """Поиск в ширину"""
    start_vert = graph.getVertex(start)
    start_vert.color = 'grey'
    start_vert.dist = 0
    start_vert.pred = None
    
    queue = deque()
    queue.append(start_vert)
    
    while queue:
        current = queue.popleft()
        for nbr in current.getConnections():
            if nbr.color == 'white':
                nbr.color = 'grey'
                nbr.dist = current.dist + 1
                nbr.pred = current
                queue.append(nbr)
        current.color = 'black'


def get_path(graph, start, end):
    """Возвращает путь от start до end"""
    bfs(graph, start)
    path = []
    current = graph.getVertex(end)
    if current is None or (current.pred is None and current.getId() != start):
        return None
    while current:
        path.append(current.getId())
        current = current.pred
    path.reverse()
    return path


def print_graph(graph):
    print("Список смежности:")
    for v in graph:
        neighbors = [nbr.getId() for nbr in v.getConnections()]
        print(f"  {v.getId()} -> {neighbors}")



g = Graph()
g.addEdge("Вы", "Карти")
g.addEdge("Вы", "Игорек")
g.addEdge("Вы", "Янг ЛИн")
g.addEdge("Игорек", "Трилл Пилл")
g.addEdge("Карти", "Трилл Пилл")
g.addEdge("Янг Лин", "Том")

print_graph(g)

path = get_path(g, "Вы", "Трилл Пилл")
print(f"\nПуть от 'Вы' до 'Трилл Пилл': {' -> '.join(path)}")

Список смежности:
  Вы -> ['Карти', 'Игорек', 'Янг ЛИн']
  Карти -> ['Трилл Пилл']
  Игорек -> ['Трилл Пилл']
  Янг ЛИн -> []
  Трилл Пилл -> []
  Янг Лин -> ['Том']
  Том -> []

Путь от 'Вы' до 'Трилл Пилл': Вы -> Карти -> Трилл Пилл


## Задание 2

Реализуйте программу, которая по заданному в виде списка смежности графу строит
обращённый граф (тоже в виде списка смежности). Обращённый граф получается
изменением направления всех рёбер исходного графа. Используйте класс Graph из
лекционных материалов.

In [3]:
def reverse_graph(graph):
    """Создаёт обращённый граф (все рёбра разворачиваются)"""
    reversed_g = Graph()
    
    # Копируем все вершины
    for v in graph:
        reversed_g.addVertex(v.getId())
    
    # Разворачиваем рёбра
    for v in graph:
        for nbr in v.getConnections():
            reversed_g.addEdge(nbr.getId(), v.getId())
    
    return reversed_g


def print_graph(graph, name="Граф"):
    print(f"\n{name}:")
    for v in graph:
        neighbors = [nbr.getId() for nbr in v.getConnections()]
        print(f"  {v.getId()} -> {neighbors}")



g = Graph()
g.addEdge("A", "B")
g.addEdge("A", "C")
g.addEdge("B", "D")
g.addEdge("C", "D")

print_graph(g, "Исходный граф")

rev = reverse_graph(g)
print_graph(rev, "Обращённый граф")


Исходный граф:
  A -> ['B', 'C']
  B -> ['D']
  C -> ['D']
  D -> []

Обращённый граф:
  A -> []
  B -> ['A']
  C -> ['A']
  D -> ['B', 'C']


## Задание 3

Пусть у вас имеется следующий рецепт блинов: одно яйцо, одна чашка блинной смеси, одна
столовая ложка растительного масла и 3/4 чашки молока. Чтобы нажарить блинов, вам
нужно разогреть сковородку, смешать вместе все ингредиенты и выливать смесь по ложке
на горячую сковороду. Когда блин начинает пузыриться, переверните его и дождитесь, пока
нижняя сторона не станет золотистой. Перед тем, как приступить к поеданию блинов,
полейте их сиропом. Весь процесс показан в виде графа.
Напишите программу, которая при помощи поиска в глубину (Depth-first search, DFS)
проведет топологическую сортировку представленного графа и выдаст вариант корректной
последовательности шагов. Например.


In [4]:
class TopologicalSort:
    def __init__(self, graph):
        self.graph = graph
        self.visited = set()
        self.stack = []
    
    def dfs(self, vertex_id):
        self.visited.add(vertex_id)
        vertex = self.graph.getVertex(vertex_id)
        
        for nbr in vertex.getConnections():
            if nbr.getId() not in self.visited:
                self.dfs(nbr.getId())
        
        self.stack.append(vertex_id)
    
    def sort(self):
        for v in self.graph.getVertices():
            if v not in self.visited:
                self.dfs(v)
        return list(reversed(self.stack))


g = Graph()

# Шаги рецепта
steps = [
    "Разогреть сковороду",
    "Смешать ингредиенты",
    "Вылить смесь на сковороду",
    "Дождаться пузырьков",
    "Перевернуть блин",
    "Дождаться золотистой корочки",
    "Полить сиропом",
    "Поедать"
]

for step in steps:
    g.addVertex(step)

# Зависимости (что должно быть сделано до)
g.addEdge("Разогреть сковороду", "Вылить смесь на сковороду")
g.addEdge("Смешать ингредиенты", "Вылить смесь на сковороду")
g.addEdge("Вылить смесь на сковороду", "Дождаться пузырьков")
g.addEdge("Дождаться пузырьков", "Перевернуть блин")
g.addEdge("Перевернуть блин", "Дождаться золотистой корочки")
g.addEdge("Дождаться золотистой корочки", "Полить сиропом")
g.addEdge("Полить сиропом", "Поедать")

ts = TopologicalSort(g)
order = ts.sort()

print("Правильная последовательность действий:")
for i, step in enumerate(order, 1):
    print(f"  {i}. {step}")

Правильная последовательность действий:
  1. Смешать ингредиенты
  2. Разогреть сковороду
  3. Вылить смесь на сковороду
  4. Дождаться пузырьков
  5. Перевернуть блин
  6. Дождаться золотистой корочки
  7. Полить сиропом
  8. Поедать


## Задание 4

У вас имеется список городов и набор автобусных маршрутов между ними. Маршруты
однонаправленные, т.е. маршрут вида «город А – город Б» означает, что можно поехать из
города А в город Б, а обратного проезда нет.
Пользователь вводит два города. Ваша программа должна определить минимальный по
количеству километров возможный путь от первого до второго города, или сообщение, что
пути между городами нет. Нужно вывести как количество километров, так и получившейся
путь.
Для решения задачи используйте алгоритм Дейкстры и класс Graph из лекционных
материалов. Список городов и маршрутов возьмите согласно вашему варианту. Расстояние
между городами найдите в интернете самостоятельно.
Также необходимо нарисовать граф вашего варианта (например, здесь
https://graphonline.ru/) и приложить его в ваш отчет. 

Вариант 3.
Города:
Мариинск, Яя, Яшкино, Юрга, Анжеро-Судженск, Кемерово, Томск.
Маршруты:
Мариинск – Яя
Мариинск – Юрга
Мариинск – Яшкино
Яя – Юрга
Яя – Анжеро-Судженск
Яшкино – Юрга
Юрга – Анжеро-Судженск
Юрга – Кемерово
Анжеро-Судженск – Кемерово
Яшкино – Томск
Юрга – Томск
Томск – Кемерово

In [12]:
import heapq

class Vertex:
    def __init__(self, key):
        self.id = key
        self.connectedTo = {}
        self.color = 'white'
        self.dist = 0
        self.pred = None
    
    def addNeighbor(self, nbr, weight=0):
        self.connectedTo[nbr] = weight
    
    def getConnections(self):
        return self.connectedTo.keys()
    
    def getId(self):
        return self.id
    
    # ДОБАВЛЯЕМ ЭТОТ МЕТОД
    def getWeight(self, nbr):
        return self.connectedTo[nbr]


class Graph:
    def __init__(self):
        self.vertList = {}
        self.numVertices = 0
    
    def addVertex(self, key):
        self.numVertices += 1
        newVertex = Vertex(key)
        self.vertList[key] = newVertex
        return newVertex
    
    def getVertex(self, n):
        return self.vertList.get(n)
    
    def addEdge(self, f, t, cost=0):
        if f not in self.vertList:
            self.addVertex(f)
        if t not in self.vertList:
            self.addVertex(t)
        self.vertList[f].addNeighbor(self.vertList[t], cost)
    
    def getVertices(self):
        return self.vertList.keys()
    
    def __iter__(self):
        return iter(self.vertList.values())


def dijkstra(graph, start, end):
    distances = {v: float('inf') for v in graph.getVertices()}
    distances[start] = 0
    parents = {start: None}
    pq = [(0, start)]
    visited = set()
    
    while pq:
        current_dist, current = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if current == end:
            break
        
        current_vertex = graph.getVertex(current)
        for neighbor in current_vertex.getConnections():
            nbr_id = neighbor.getId()
            if nbr_id in visited:
                continue
            
            new_dist = current_dist + current_vertex.getWeight(neighbor)
            if new_dist < distances[nbr_id]:
                distances[nbr_id] = new_dist
                parents[nbr_id] = current
                heapq.heappush(pq, (new_dist, nbr_id))
    
    if distances[end] == float('inf'):
        return None, None
    
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = parents.get(current)
    path.reverse()
    
    return distances[end], path


def print_graph(graph):
    print("Список смежности (вершина -> [сосед(расстояние)]):")
    for v in graph:
        neighbors = [(nbr.getId(), v.getWeight(nbr)) for nbr in v.getConnections()]
        print(f"  {v.getId()} -> {neighbors}")



g = Graph()

# Города
cities = ["Мариинск", "Яя", "Яшкино", "Юрга", "Анжеро-Судженск", "Кемерово", "Томск"]

for city in cities:
    g.addVertex(city)

# Маршруты с расстояниями
routes = [
    ("Мариинск", "Яя", 106),
    ("Мариинск", "Юрга", 131),
    ("Мариинск", "Яшкино", 140),
    ("Яя", "Юрга", 105),
    ("Яя", "Анжеро-Судженск", 90),
    ("Яшкино", "Юрга", 47),
    ("Юрга", "Анжеро-Судженск", 137),
    ("Юрга", "Кемерово", 140),
    ("Анжеро-Судженск", "Кемерово", 103),
    ("Яшкино", "Томск", 165),
    ("Юрга", "Томск", 107),
    ("Томск", "Кемерово", 205),
]

for f, t, dist in routes:
    g.addEdge(f, t, dist)

print_graph(g)

start = input("Введите начальный город: ")
end = input("Введите конечный город: ")

distance, path = dijkstra(g, start, end)

if distance is None:
    print(f"Пути из '{start}' в '{end}' не существует!")
else:
    print(f"Кратчайший путь: {' -> '.join(path)}")
    print(f"Расстояние: {distance} км")

Список смежности (вершина -> [сосед(расстояние)]):
  Мариинск -> [('Яя', 106), ('Юрга', 131), ('Яшкино', 140)]
  Яя -> [('Юрга', 105), ('Анжеро-Судженск', 90)]
  Яшкино -> [('Юрга', 47), ('Томск', 165)]
  Юрга -> [('Анжеро-Судженск', 137), ('Кемерово', 140), ('Томск', 107)]
  Анжеро-Судженск -> [('Кемерово', 103)]
  Кемерово -> []
  Томск -> [('Кемерово', 205)]
Кратчайший путь: Мариинск -> Юрга -> Кемерово
Расстояние: 271 км
